In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import (
	AutoTokenizer,
	AutoModelForSequenceClassification,
	get_linear_schedule_with_warmup,
)
from tqdm import tqdm
import re

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [2]:
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(train_df['label'].value_counts().sort_index())
train_df.head(3)

Train: 252000 rows | Test: 168000 rows
label
0    50400
1    50400
2    50400
3    50400
4    50400
Name: count, dtype: int64


,id,sentence,label
0,0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,4
1,1,Passt wie angegossen\n\nTasche kam schnell und...,4
2,2,Schlechte Qualität\n\nIch habe sehr lange auf ...,1


In [3]:
def clean_text(text):
	text = re.sub(r'\s+', ' ', text).strip()
	return text

train_df['sentence'] = train_df['sentence'].apply(clean_text)
test_df['sentence']  = test_df['sentence'].apply(clean_text)

In [4]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(
	train_df['sentence'].tolist(),
	train_df['label'].tolist(),
	test_size=0.10,
	random_state=SEED,
	stratify=train_df['label'].tolist()
)
print(f'Train split: {len(train_sentences)} | Val split: {len(val_sentences)}')

Train split: 226800 | Val split: 25200


In [5]:
class ReviewsDataset(Dataset):
	def __init__(self, encodings, labels):
		self.encodings = encodings
		self.labels = labels

	def __getitem__(self, idx):
		item = {key: val[idx].clone() for key, val in self.encodings.items()}
		item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
		return item

	def __len__(self):
		return len(self.labels)

In [6]:
BERT_MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'
BERT_MODEL_PATH = 'checkpoints/bert_full_best_model_init_num_epochs-5lr-1e-06step_size-Nonegamma-Nonetrain_size-180000val_size-10000.pth'

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

os.makedirs('data/bert-encodings', exist_ok=True)

if not os.path.exists('data/bert-encodings/bert_val_encodings.pt'):
	bert_val_enc  = bert_tokenizer(val_sentences, truncation=True, padding=True, max_length=256, return_tensors='pt')
	bert_test_enc = bert_tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=256, return_tensors='pt')
	torch.save(bert_val_enc,  'data/bert-encodings/bert_val_encodings.pt')
	torch.save(bert_test_enc, 'data/bert-encodings/bert_test_encodings.pt')
else:
	bert_val_enc  = torch.load('data/bert-encodings/bert_val_encodings.pt',  weights_only=False)
	bert_test_enc = torch.load('data/bert-encodings/bert_test_encodings.pt', weights_only=False)

bert_val_dataset  = ReviewsDataset(bert_val_enc,  val_labels)
bert_test_dataset = ReviewsDataset(bert_test_enc, [-1] * len(test_df))

bert_val_loader  = DataLoader(bert_val_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
bert_test_loader = DataLoader(bert_test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=5)
bert_model.load_state_dict(torch.load(BERT_MODEL_PATH, map_location=device))
bert_model = bert_model.to(device)
bert_model.eval()
print('BERT model loaded.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT model loaded.


In [7]:
ROBERTA_MODEL_NAME = 'xlm-roberta-large'
ROBERTA_MODEL_PATH = 'checkpoints/roberta_best_model_init_num_epochs-5%lr-1e-06%step_size-None%gamma-None%train_size-226800%val_size-25200.pth'

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)

os.makedirs('data/roberta-encodings', exist_ok=True)

if not os.path.exists('data/roberta-encodings/roberta_val_encodings.pt'):
	roberta_val_enc  = roberta_tokenizer(val_sentences, truncation=True, padding=True, max_length=256, return_tensors='pt')
	roberta_test_enc = roberta_tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=256, return_tensors='pt')
	torch.save(roberta_val_enc,  'data/roberta-encodings/roberta_val_encodings.pt')
	torch.save(roberta_test_enc, 'data/roberta-encodings/roberta_test_encodings.pt')
else:
	roberta_val_enc  = torch.load('data/roberta-encodings/roberta_val_encodings.pt',  weights_only=False)
	roberta_test_enc = torch.load('data/roberta-encodings/roberta_test_encodings.pt', weights_only=False)

roberta_val_dataset  = ReviewsDataset(roberta_val_enc,  val_labels)
roberta_test_dataset = ReviewsDataset(roberta_test_enc, [-1] * len(test_df))

roberta_val_loader  = DataLoader(roberta_val_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
roberta_test_loader = DataLoader(roberta_test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL_NAME, num_labels=5)
roberta_model.load_state_dict(torch.load(ROBERTA_MODEL_PATH, map_location=device))
roberta_model = roberta_model.to(device)
roberta_model.eval()
print('RoBERTa model loaded.')

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

c:\Users\vlad\anaconda3\envs\all\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vlad\.cache\huggingface\hub\models--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model loaded.


In [ ]:
def predict_probs(model, dataloader):
	model.eval()
	all_probs = []
	with torch.no_grad():
		for batch in tqdm(dataloader):
			batch = {k: v.to(device) for k, v in batch.items()}
			batch.pop('labels')
			outputs = model(**batch)
			probs = F.softmax(outputs.logits, dim=1)
			all_probs.append(probs.cpu().numpy())
	return np.vstack(all_probs) #(N, 5)


bert_probs_val  = predict_probs(bert_model, bert_val_loader)
roberta_probs_val = predict_probs(roberta_model, roberta_val_loader)

bert_probs_test   = predict_probs(bert_model, bert_test_loader)
roberta_probs_test = predict_probs(roberta_model, roberta_test_loader)

print(f'bert_probs_val shape:    {bert_probs_val.shape}')
print(f'roberta_probs_val shape: {roberta_probs_val.shape}')

  0%|          | 0/1575 [00:00<?, ?it/s]

In [ ]:
class StackingMLP(nn.Module):
	def __init__(self, input_dim=10, hidden_dim=64, num_classes=5):
		super().__init__()
		self.mlp = nn.Sequential(
			nn.Linear(input_dim, hidden_dim), # input_dim = 5 classes * 2 models
			nn.ReLU(),
			nn.Dropout(0.1),
			nn.Linear(hidden_dim, num_classes)
		)

	def forward(self, x):
		return self.mlp(x)

In [ ]:
X_val = np.hstack([bert_probs_val, roberta_probs_val])
y_val = np.array(val_labels)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)

hyperparams_stack = {
	'num_epochs': 100,
	'lr': 1e-3,
	'hidden_dim': 64,
}

stacking_model = StackingMLP(input_dim=10, hidden_dim=hyperparams_stack['hidden_dim'], num_classes=5).to(device)
optimizer_stack = torch.optim.Adam(stacking_model.parameters(), lr=hyperparams_stack['lr'])
criterion = nn.CrossEntropyLoss()

best_stack_score = 0.0

for epoch in range(hyperparams_stack['num_epochs']):
	stacking_model.train()
	optimizer_stack.zero_grad()
	logits = stacking_model(X_val_tensor)
	loss = criterion(logits, y_val_tensor)
	loss.backward()
	optimizer_stack.step()

	stacking_model.eval()
	with torch.no_grad():
		preds = torch.argmax(stacking_model(X_val_tensor), dim=1).cpu().numpy()
	mae = np.mean(np.abs(preds - y_val))
	score = 1.0 - mae / 4.0

	if score > best_stack_score:
		best_stack_score = score
		torch.save(stacking_model.state_dict(), 'checkpoints/best_stacking_mlp.pth')

	print(f'Epoch {epoch+1:3d} | loss={loss.item():.4f} | val_score={score:.4f}')

print(f'\nBest stacking val score: {best_stack_score:.4f}')

In [ ]:
stacking_model.load_state_dict(torch.load('checkpoints/best_stacking_mlp.pth', map_location=device))
stacking_model.eval()

X_test = np.hstack([bert_probs_test, roberta_probs_test])  # (N_test, 10)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():
	final_preds = torch.argmax(stacking_model(X_test_tensor), dim=1).cpu().numpy()

submission_df = pd.DataFrame({
	'id'   : test_df['id'].astype(int),
	'label': final_preds.astype(int)
})

assert len(submission_df) == len(test_df)
assert submission_df['label'].isin([0,1,2,3,4]).all()

submission_df.to_csv('data/submission_stacking.csv', index=False)
print('Submission saved.')
print(submission_df.head(5))